In [1]:
!pip install semantic-kernel
!pip install openai --upgrade
import os
from typing import Annotated
from openai import AsyncOpenAI

Error processing line 1 of /opt/anaconda3/envs/AgentsIA/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 195, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Error processing line 1 of /opt/anaconda3/envs/AgentsIA/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 195, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from dotenv import load_dotenv

In [1]:
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

In [4]:
import random

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [11]:
load_dotenv()
client = AsyncOpenAI(
    #api_key=os.environ.get("GITHUB_TOKEN"),
    api_key= "", #Mi token de GitHub
    base_url="",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

In [16]:
agent = ChatCompletionAgent(
    service=chat_completion_service,
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    #instructions="Tú eres un poderoso agente AI que puede ayudarme a planear mis vacaciones a destinos random de Europa",
    instructions="You are a powerful AI agent that can help me plan my vacations to random destinations in Europe",
)

Ejecutar el Agente

Ahora podemos ejecutar el Agente definiendo un hilo del tipo ChatHistoryAgentThread. Cualquier mensaje del sistema necesario se proporciona al argumento de palabra clave messages de invoke_stream del agente.

Una vez definidos, creamos un user_inputs, que será lo que el usuario envíe al agente. En este caso, hemos configurado este mensaje como Planéame unas vacaciones soleadas.

Siéntete libre de cambiar este mensaje para ver cómo responde el agente de manera diferente.

In [18]:
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        #"Organizame un viaje seleccionando un destino al azar en Europa, organiza jornadas y restaurante",
        #"plan me a day trip"
        "planea un día de mi viaje",
    ]

    for user_input in user_inputs:
        print(f"# User: {user_input}\n")
        first_chunk = True
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            # 5. Print the response
            if first_chunk:
                print(f"# {response.name}: ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread
        print()

    # Clean up the thread
    await thread.delete() if thread else None

await main()

# User: planea un día de mi viaje

# TravelAgent: Parece que he recibido un destino un poco inesperado, ¡pues es Tokio, Japón! Sin embargo, si prefieres un destino en Europa, puedo buscar otro lugar. Déjame saber si deseas que continúe con Tokio o si prefieres un destino europeo.


In [ ]:
async def main():
    thread: ChatHistoryAgentThread | None = None

    print("Chat con TravelAgent (escribir 'salir' para terminar)\n")

    while True:

        user_input = input("# User: ")

        if user_input.lower() in ["salir", "exit", "quit", "bye"]:
            print("¡Hasta luego!")
            break

        if not user_input.strip():
            continue

        first_chunk = True
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            if first_chunk:
                print(f"\n# {response.name}: ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread

        print("\n")

    # Limpiar thread al salir
    await thread.delete() if thread else None

await main()

Chat con TravelAgent (escribir 'salir' para terminar)

# User: planea un dia de mi viaje

# TravelAgent: Parece que he obtenido un destino que no es europeo, ¡pero no te preocupes! Voy a buscar un destino en Europa para ayudarte a planear tu día de viaje. 

Por favor, dame un momento.Parece que las opciones que estoy generando no son destinos en Europa. Permíteme intentarlo de nuevo para encontrar un destino europeo y luego planear un día de actividades. Un momento, por favor.

# User: que tal paris?

# TravelAgent: ¡Perfecto! Vamos a planear un día en París. Aquí tienes un itinerario sugerido para disfrutar de la ciudad:

### Itinerario de un Día en París

**Mañana:**
1. **Desayuno en una cafetería local:** Comienza el día con un delicioso croissant y un café en una de las clásicas cafeterías parisinas como Café de Flore o Les Deux Magots.
   
2. **Visita a la Torre Eiffel:** Dirígete a la Torre Eiffel y disfrútala desde el Champ de Mars. Si prefieres, puedes subir a la cima para disf